# Model 2: ResNet50 — Feature Extraction + Fine-Tuning
### Retinal OCT Disease Classification (Kermany 2018 Dataset)

**Strategy:** Two-phase training
- Phase 1: Freeze ResNet50 base → train classifier head only (Feature Extraction)
- Phase 2: Unfreeze top layers → fine-tune the whole model

**Challenges Addressed:** Class Imbalance, Overfitting, Speckle Noise, Inter-Class Similarity

In [ ]:
# ─────────────────────────────────────────────
# SECTION 1: IMPORTS
# ─────────────────────────────────────────────
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    classification_report, confusion_matrix,
    cohen_kappa_score, roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize
import cv2
import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 2: CONFIGURATION
# ─────────────────────────────────────────────
DATASET_PATH = './OCT2017'
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS_FE    = 15   # Phase 1: Feature Extraction
EPOCHS_FT    = 20   # Phase 2: Fine-Tuning
SEED         = 42
NUM_CLASSES  = 4
CLASS_NAMES  = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
VAL_DIR   = os.path.join(DATASET_PATH, 'val')
TEST_DIR  = os.path.join(DATASET_PATH, 'test')

tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 3: CLASS WEIGHTS (Imbalance Fix)
# ─────────────────────────────────────────────
def get_class_weights(train_dir, class_names):
    counts  = [len(os.listdir(os.path.join(train_dir, c))) for c in class_names]
    total   = sum(counts)
    weights = {i: total / (len(class_names) * c) for i, c in enumerate(counts)}
    print('Class counts :', dict(zip(class_names, counts)))
    print('Class weights:', weights)
    return weights

class_weights = get_class_weights(TRAIN_DIR, CLASS_NAMES)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 4: DATA GENERATORS
# ResNet50 expects ImageNet-normalised inputs
# ─────────────────────────────────────────────
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

def add_gaussian_noise(image):
    noise = np.random.normal(0, 0.01, image.shape)
    return np.clip(image + noise, 0, 255)

train_datagen = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(preprocessing_function=resnet_preprocess)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', seed=SEED, shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', seed=SEED, shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', seed=SEED, shuffle=False
)
print('Class indices:', train_gen.class_indices)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 5: BUILD RESNET50 MODEL
# Phase 1 — Freeze base, train head only
# ─────────────────────────────────────────────
def build_resnet50_model(num_classes=4, trainable_base=False):
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = trainable_base

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name='ResNet50_OCT')
    return model, base_model

resnet_model, base_model = build_resnet50_model(trainable_base=False)
resnet_model.summary()

In [ ]:
# ─────────────────────────────────────────────
# SECTION 6: PHASE 1 — FEATURE EXTRACTION
# Base is frozen, only classifier head trains
# ─────────────────────────────────────────────
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_fe = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint('best_resnet_fe.keras', monitor='val_accuracy', save_best_only=True)
]

print('\n=== PHASE 1: Feature Extraction ===')
history_fe = resnet_model.fit(
    train_gen,
    epochs=EPOCHS_FE,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_fe,
    verbose=1
)
print(f'Phase 1 complete. Best val accuracy: {max(history_fe.history["val_accuracy"])*100:.2f}%')

In [ ]:
# ─────────────────────────────────────────────
# SECTION 7: PHASE 2 — FINE-TUNING
# Unfreeze top 30 layers of ResNet50 base
# Use much smaller learning rate to avoid destroying pretrained weights
# ─────────────────────────────────────────────
base_model.trainable = True

# Freeze all layers except the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_count = sum(1 for l in resnet_model.layers if l.trainable)
print(f'Trainable layers after unfreeze: {trainable_count}')

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # Low LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_ft = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint('best_resnet_ft.keras', monitor='val_accuracy', save_best_only=True)
]

print('\n=== PHASE 2: Fine-Tuning ===')
history_ft = resnet_model.fit(
    train_gen,
    epochs=EPOCHS_FT,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_ft,
    verbose=1
)
print(f'Phase 2 complete. Best val accuracy: {max(history_ft.history["val_accuracy"])*100:.2f}%')

In [ ]:
# ─────────────────────────────────────────────
# SECTION 8: PLOT COMBINED HISTORY
# ─────────────────────────────────────────────
def plot_combined_history(h1, h2, model_name='ResNet50'):
    acc  = h1.history['accuracy']  + h2.history['accuracy']
    val  = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss'] + h2.history['loss']
    vloss= h1.history['val_loss'] + h2.history['val_loss']
    split_point = len(h1.history['accuracy'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, train, val_, title in zip(
            axes, [acc, loss], [val, vloss],
            ['Accuracy', 'Loss']):
        ax.plot(train, label=f'Train {title}')
        ax.plot(val_,  label=f'Val {title}')
        ax.axvline(split_point, color='red', linestyle='--', label='Fine-tune start')
        ax.set_title(f'{model_name} — {title}')
        ax.legend(); ax.grid(True)

    plt.tight_layout()
    plt.savefig(f'{model_name}_training_curves.png', dpi=150)
    plt.show()

plot_combined_history(history_fe, history_ft)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 9: FULL EVALUATION
# ─────────────────────────────────────────────
def evaluate_model(model, test_gen, class_names, model_name):
    test_gen.reset()
    y_pred_prob = model.predict(test_gen, verbose=1)
    y_pred      = np.argmax(y_pred_prob, axis=1)
    y_true      = test_gen.classes

    acc = np.mean(y_pred == y_true)
    print(f'\n{model_name} — Test Accuracy: {acc*100:.2f}%')
    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} — Confusion Matrix', fontweight='bold')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'{model_name.replace(" ","_")}_cm.png', dpi=150)
    plt.show()

    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"Cohen's Kappa: {kappa:.4f}")

    y_true_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    plt.figure(figsize=(8,6))
    auc_scores = {}
    for i, cls in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true_bin[:,i], y_pred_prob[:,i])
        auc = roc_auc_score(y_true_bin[:,i], y_pred_prob[:,i])
        auc_scores[cls] = auc
        plt.plot(fpr, tpr, label=f'{cls} (AUC={auc:.3f})')
    plt.plot([0,1],[0,1],'k--')
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title(f'{model_name} — AUC-ROC', fontweight='bold')
    plt.legend(); plt.grid(True)
    plt.tight_layout()
    plt.savefig(f'{model_name.replace(" ","_")}_roc.png', dpi=150)
    plt.show()
    print('AUC per class:', auc_scores)

    return {'accuracy': acc, 'kappa': kappa, 'auc': auc_scores,
            'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_pred_prob}

resnet_results = evaluate_model(resnet_model, test_gen, CLASS_NAMES, 'ResNet50')

In [ ]:
# ─────────────────────────────────────────────
# SECTION 10: GRAD-CAM on ResNet50
# ─────────────────────────────────────────────
import cv2

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index    = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_index.numpy()

def display_gradcam(model, test_gen, class_names, last_conv='conv5_block3_out'):
    """conv5_block3_out is the last conv layer in ResNet50"""
    from tensorflow.keras.applications.resnet50 import preprocess_input
    test_gen.reset()
    images_raw, labels = next(test_gen)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i, ax in enumerate(axes.flat):
        img = images_raw[i]
        input_img = np.expand_dims(img, axis=0)
        heatmap, pred_idx = make_gradcam_heatmap(input_img, model, last_conv)
        # Reverse ResNet preprocess to visualise
        display_img = img.copy()
        display_img = (display_img - display_img.min()) / (display_img.max() - display_img.min() + 1e-8)
        heatmap_resized = cv2.resize(heatmap, IMG_SIZE)
        heatmap_rgb = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_rgb = cv2.cvtColor(heatmap_rgb, cv2.COLOR_BGR2RGB) / 255.0
        overlay = 0.6 * display_img + 0.4 * heatmap_rgb
        ax.imshow(np.clip(overlay, 0, 1))
        true_lbl = class_names[np.argmax(labels[i])]
        pred_lbl = class_names[pred_idx]
        color = 'green' if true_lbl == pred_lbl else 'red'
        ax.set_title(f'True:{true_lbl}\nPred:{pred_lbl}', color=color, fontsize=9)
        ax.axis('off')
    plt.suptitle('Grad-CAM Heatmaps — ResNet50', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('resnet50_gradcam.png', dpi=150)
    plt.show()

display_gradcam(resnet_model, test_gen, CLASS_NAMES)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 11: SAVE MODEL
# ─────────────────────────────────────────────
resnet_model.save('resnet50_finetuned_final.keras')
print('ResNet50 model saved.')
print(f"\n=== SUMMARY ===")
print(f"Test Accuracy : {resnet_results['accuracy']*100:.2f}%")
print(f"Cohen's Kappa : {resnet_results['kappa']:.4f}")
for cls, auc in resnet_results['auc'].items():
    print(f"AUC [{cls}]     : {auc:.4f}")